# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted)

## Get entities LLM

In [ ]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_PreLabel.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]


### no boost

In [ ]:
#%%
# Wrap original extract_llm_entities with a progress bar (no need to modify .py)
def extract_llm_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="LLM Prediction"):
        result = extract_llm_entities([sent])[0]
        results.append(result)
    return results

In [2]:
#%%
# Run prediction with progress bar
pred_results = extract_llm_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 445/445 [28:07<00:00,  3.79s/it]


### boost

In [2]:
from tqdm import tqdm

BATCH_SIZE = 200  # 可視 rate limit 調 100~500

def extract_llm_entities_in_batches(sentences, batch_size=BATCH_SIZE):
    results = []
    n = len(sentences)
    for start in tqdm(range(0, n, batch_size), desc="LLM Prediction"):
        batch = sentences[start:start+batch_size]
        # 一次丟一大批，讓 extract_llm_entities 內部開並行
        results.extend(extract_llm_entities(batch))
    return results

# 使用
pred_results = extract_llm_entities_in_batches(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 3/3 [02:21<00:00, 47.03s/it]


### evaluate

In [3]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [4]:
#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.6103983794733289
Macro F1: 0.5016426737227713
Weighted F1: 0.6114972944510506

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.66      0.64      0.65        77
           Interest Rates       0.75      0.55      0.64        49
                Inflation       0.95      0.92      0.93        85
               Employment       0.74      0.79      0.76        33
             Unemployment       0.89      1.00      0.94         8
                      GDP       0.48      0.46      0.47        26
                    Trade       0.75      1.00      0.86         6
                 Congress       0.00      0.00      0.00         0
          Monetary Policy       0.59      0.70      0.64        70
      Financial Stability       0.00      0.00      0.00         0
          Price Stability       0.64      0.41      0.50        17
Regulatory Implementation       0.00      0.00      0.00         2
              

# Evaluation Sentiment Analysis

In [7]:
# eval_sentiment_both.py
# Evaluate sentiment scoring (nlp + llm) on the holdout set using gold entities.
# Cleaner version with better debugging and data handling.

import ast
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    classification_report,
)

from get_sentiment_nlp import extract_nlp_sentiment
from get_sentiment_llm import extract_llm_sentiment

HOLDOUT_CSV = "holdout_eval_set.csv"

# ---------------------------
# Parse the "Entities" cell -> list of (entity, score) tuples
def parse_entities_cell(s):
    """Parse string representation of entity list into (entity, score) tuples."""
    if pd.isna(s) or s.strip() == "":
        return []
    
    try:
        raw = ast.literal_eval(s.strip())
    except Exception as e:
        print(f"Warning: Could not parse entities: {s[:50]}... Error: {e}")
        return []
    
    entities = []
    if not isinstance(raw, list):
        return []
        
    for item in raw:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue
        entity_name, score = item[0], item[1]
        
        # Skip empty entities
        if not entity_name or entity_name.strip() == "":
            continue
            
        # Convert score to float
        try:
            score = float(score)
            entities.append((str(entity_name).strip(), score))
        except (ValueError, TypeError):
            print(f"Warning: Invalid score for entity '{entity_name}': {score}")
            continue
    
    return entities

# ---------------------------
# Load and prepare holdout data
def load_holdout_data(csv_path):
    """Load holdout data and return structured format."""
    print(f"Loading holdout data from {csv_path}")
    
    # Read CSV - assuming columns are: Sentence, Entities, Source
    df = pd.read_csv(csv_path)
    print(f"Raw CSV shape: {df.shape}")
    
    # Rename columns if needed (adjust based on your actual CSV structure)
    expected_cols = ['Sentence', 'Entities']
    if not all(col in df.columns for col in expected_cols):
        print(f"Warning: Expected columns {expected_cols}, got {list(df.columns)}")
        # If your CSV has different column names, adjust here:
        # df = df.rename(columns={'YourSentenceCol': 'Sentence', 'YourEntitiesCol': 'Entities'})
    
    # Clean and validate data
    df = df.dropna(subset=['Sentence']).copy()
    df['Sentence'] = df['Sentence'].str.strip()
    df = df[df['Sentence'].str.len() > 0].copy()
    
    print(f"After cleaning: {len(df)} sentences")
    
    # Parse entities and create gold standard
    gold_data = []
    sentence_entity_counts = defaultdict(lambda: defaultdict(int))
    
    for idx, row in df.iterrows():
        sentence = row['Sentence']
        entities = parse_entities_cell(row.get('Entities', '[]'))
        
        if not entities:
            continue
            
        # Check for duplicates within this sentence
        entity_names = [ent for ent, _ in entities]
        for ent_name in entity_names:
            sentence_entity_counts[sentence][ent_name] += 1
        
        # Add to gold data with position index
        for pos, (entity_name, true_score) in enumerate(entities):
            gold_data.append({
                'sentence': sentence,
                'position': pos,
                'entity': entity_name,
                'true_score': true_score,
                'source_row': idx
            })
    
    # Report any duplicate entities within sentences
    duplicates_found = False
    for sentence, entity_counts in sentence_entity_counts.items():
        for entity, count in entity_counts.items():
            if count > 1:
                if not duplicates_found:
                    print("\n=== DUPLICATE ENTITIES FOUND ===")
                    duplicates_found = True
                print(f"Sentence: '{sentence[:80]}...'")
                print(f"  Entity '{entity}' appears {count} times")
    
    if not duplicates_found:
        print("✓ No duplicate entities found within sentences")
    
    gold_df = pd.DataFrame(gold_data)
    print(f"Gold standard: {len(gold_df)} entity annotations across {gold_df['sentence'].nunique()} sentences")
    
    return gold_df

# ---------------------------
# Prepare data for prediction methods
def prepare_prediction_inputs(gold_df):
    """Prepare input formats for both NLP and LLM methods."""
    # Group by sentence to get unique sentences and their entities
    sentence_groups = gold_df.groupby('sentence').apply(
        lambda x: x.sort_values('position')[['entity', 'true_score']].values.tolist()
    ).to_dict()
    
    print(f"Preparing inputs for {len(sentence_groups)} unique sentences")
    
    # Format for LLM method
    llm_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entities = [{"name": ent} for ent, _ in entity_data]
        llm_inputs.append({
            "sentence": sentence,
            "entities": entities
        })
    
    # Format for NLP method  
    nlp_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entity_names = [ent for ent, _ in entity_data]
        nlp_inputs.append((sentence, entity_names))
    
    return llm_inputs, nlp_inputs

# ---------------------------
# Convert prediction results to standardized DataFrame
def predictions_to_dataframe(predictions, method_name):
    """Convert prediction results to DataFrame with sentence/position/entity/score."""
    rows = []
    
    if method_name == "LLM":
        # LLM format: [{"sentence": str, "entities": [{"name": str, "sentiment": float}, ...]}]
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    
    elif method_name == "NLP":
        # NLP format: [{"sentence": str, "entities": [{"name": str, "sentiment": float}, ...]}]
        # (Assuming same format as LLM - adjust if your NLP method returns different format)
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    
    df = pd.DataFrame(rows)
    print(f"{method_name} predictions: {len(df)} entity predictions")
    return df

# ---------------------------
# Compute comprehensive metrics
def compute_metrics(method_name, gold_df, pred_df, score_to_label):
    """Compute and print comprehensive evaluation metrics."""
    print(f"\n{'='*50}")
    print(f"EVALUATING: {method_name}")
    print(f"{'='*50}")
    
    # Merge gold and predictions
    merged = gold_df.merge(
        pred_df, 
        on=['sentence', 'position', 'entity'], 
        how='left',
        suffixes=('', '_pred')
    )
    
    print(f"Gold annotations: {len(gold_df)}")
    print(f"Predictions: {len(pred_df)}")
    print(f"Matched: {len(merged[~merged['pred_score'].isna()])}")
    print(f"Unmatched: {len(merged[merged['pred_score'].isna()])}")
    
    # Filter to valid predictions
    valid = merged[
        (~merged['pred_score'].isna()) & 
        (merged['pred_score'] != "") &
        (merged['pred_score'] != "")
    ].copy()
    
    if len(valid) == 0:
        print("❌ No valid predictions to evaluate!")
        return
    
    # Convert scores to classification labels
    def score_to_class(score):
        try:
            return score_to_label.get(float(score), None)
        except:
            return None
    
    valid['true_label'] = valid['true_score'].apply(score_to_class)
    valid['pred_label'] = valid['pred_score'].apply(score_to_class)
    
    # Filter to valid labels
    valid_labels = valid[
        (~valid['true_label'].isna()) & 
        (~valid['pred_label'].isna())
    ].copy()
    
    if len(valid_labels) == 0:
        print("❌ No valid label pairs after classification mapping!")
        return
    
    print(f"Valid for classification: {len(valid_labels)}")
    
    # Classification metrics
    y_true = valid_labels['true_label'].astype(int).values
    y_pred = valid_labels['pred_label'].astype(int).values
    labels = sorted(set(score_to_label.values()))
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', labels=labels, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0)
    
    # Regression metrics (MAE)
    y_true_reg = valid['true_score'].astype(float).values
    y_pred_reg = valid['pred_score'].astype(float).values
    mae = mean_absolute_error(y_true_reg, y_pred_reg)
    
    # Baselines
    majority_score = pd.Series(y_true_reg).mode().iloc[0]
    median_score = np.median(y_true_reg)
    mae_majority = mean_absolute_error(y_true_reg, [majority_score] * len(y_true_reg))
    mae_median = mean_absolute_error(y_true_reg, [median_score] * len(y_true_reg))
    
    # Print results
    print(f"\n--- OVERALL METRICS (n={len(valid_labels)}) ---")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Macro F1:           {macro_f1:.4f}")
    print(f"Micro F1:           {micro_f1:.4f}")
    print(f"Weighted F1:        {weighted_f1:.4f}")
    print(f"MAE:                {mae:.4f}")
    print(f"MAE (majority={majority_score:.2f}): {mae_majority:.4f} (Δ={mae_majority-mae:+.4f})")
    print(f"MAE (median={median_score:.2f}):   {mae_median:.4f} (Δ={mae_median-mae:+.4f})")
    
    # Per-entity breakdown
    print(f"\n--- PER-ENTITY BREAKDOWN ---")
    entity_stats = []
    for entity in sorted(valid_labels['entity'].unique()):
        entity_data = valid_labels[valid_labels['entity'] == entity]
        if len(entity_data) < 2:  # Skip entities with too few samples
            continue
            
        ent_y_true = entity_data['true_label'].astype(int).values
        ent_y_pred = entity_data['pred_label'].astype(int).values
        
        ent_acc = accuracy_score(ent_y_true, ent_y_pred)
        ent_f1 = f1_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        
        entity_stats.append((entity, len(entity_data), ent_acc, ent_f1))
    
    # Sort by count desc, then F1 desc
    entity_stats.sort(key=lambda x: (-x[1], -x[3]))
    
    for entity, count, acc, f1 in entity_stats:
        print(f"{entity:<25} n={count:3d}  acc={acc:.3f}  F1={f1:.3f}")
    
    # Classification report
    print(f"\n--- CLASSIFICATION REPORT ---")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

# ---------------------------
# Main execution
def main():
    print("🚀 Starting Sentiment Evaluation")
    
    # Load score to label mapping
    try:
        with open("score_to_label.json", "r") as f:
            score_to_label_raw = json.load(f)
        score_to_label = {float(k): int(v) for k, v in score_to_label_raw.items()}
        print(f"✓ Loaded score-to-label mapping: {len(score_to_label)} mappings")
    except FileNotFoundError:
        print("❌ score_to_label.json not found!")
        return
    
    # Load holdout data
    try:
        gold_df = load_holdout_data(HOLDOUT_CSV)
    except FileNotFoundError:
        print(f"❌ Holdout file not found: {HOLDOUT_CSV}")
        return
    
    if len(gold_df) == 0:
        print("❌ No valid gold data found!")
        return
    
    # Prepare inputs for prediction methods
    llm_inputs, nlp_inputs = prepare_prediction_inputs(gold_df)
    
    # Run predictions
    print(f"\n🔮 Running LLM predictions...")
    try:
        llm_predictions = extract_llm_sentiment(llm_inputs)
        llm_pred_df = predictions_to_dataframe(llm_predictions, "LLM")
    except Exception as e:
        print(f"❌ LLM prediction failed: {e}")
        llm_pred_df = pd.DataFrame()
    
    print(f"\n🔍 Running NLP predictions...")
    try:
        nlp_predictions = extract_nlp_sentiment(nlp_inputs)
        nlp_pred_df = predictions_to_dataframe(nlp_predictions, "NLP")
    except Exception as e:
        print(f"❌ NLP prediction failed: {e}")
        nlp_pred_df = pd.DataFrame()
    
    # Evaluate both methods
    if not llm_pred_df.empty:
        compute_metrics("LLM", gold_df, llm_pred_df, score_to_label)
    
    if not nlp_pred_df.empty:
        compute_metrics("NLP", gold_df, nlp_pred_df, score_to_label)
    
    print(f"\n✅ Evaluation complete!")

if __name__ == "__main__":
    main()

2025-08-26 12:04:48,990 | INFO | [extract] received items=288 | with_entities=288 | skipped=0
2025-08-26 12:04:48,990 | INFO | [extract] batching | chunks=15 | chunk_size≈20


🚀 Starting Sentiment Evaluation
✓ Loaded score-to-label mapping: 7 mappings
Loading holdout data from holdout_eval_set.csv
Raw CSV shape: (444, 3)
After cleaning: 444 sentences

=== DUPLICATE ENTITIES FOUND ===
Sentence: 'Total PCE price inflation was 6.0 percent over the 12 months ending in October, ...'
  Entity 'Inflation' appears 2 times
Sentence: 'Core PCE price inflation, which excludes changes in con- sumer energy prices and...'
  Entity 'Inflation' appears 3 times
Sentence: 'Against this backdrop, all participants agreed that it was appropriate to raise ...'
  Entity 'Federal Reserve' appears 2 times
Sentence: 'In assessing the appropriate stance of monetary policy, the Committee will conti...'
  Entity 'Federal Reserve' appears 4 times
Sentence: 'In assessing the appropriate stance of monetary policy, the Committee will conti...'
  Entity 'Monetary Policy' appears 4 times
Sentence: 'In assessing the appropriate stance of monetary policy, the Committee will conti...'
  Entity '

2025-08-26 12:04:49,415 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:04:49,415 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:04:49,416 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:04:49,826 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:05:09,481 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:05:10,095 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:05:11,853 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:05:12,946 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 12:05:27,094 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 

LLM predictions: 776 entity predictions

🔍 Running NLP predictions...
NLP predictions: 776 entity predictions

EVALUATING: LLM
Gold annotations: 776
Predictions: 776
Matched: 735
Unmatched: 41
Valid for classification: 735

--- OVERALL METRICS (n=735) ---
Accuracy:           0.2612
Macro F1:           0.1393
Micro F1:           0.2612
Weighted F1:        0.1916
MAE:                0.3531
MAE (majority=0.33): 0.3603 (Δ=+0.0072)
MAE (median=0.33):   0.3603 (Δ=+0.0072)

--- PER-ENTITY BREAKDOWN ---
Economic Outlook          n=111  acc=0.333  F1=0.180
Federal Reserve           n=101  acc=0.248  F1=0.057
Inflation                 n= 82  acc=0.207  F1=0.097
Monetary Policy           n= 54  acc=0.241  F1=0.069
Interest Rates            n= 38  acc=0.158  F1=0.080
Employment                n= 30  acc=0.333  F1=0.210
Uncertain                 n= 26  acc=0.231  F1=0.095
Labor Market              n= 24  acc=0.208  F1=0.124
Price Stability           n= 24  acc=0.250  F1=0.114
GDP                   